<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_43_ai_dev_tools/note_lesson_43_ai_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Урок 43 — AI-інструменти розробника: ризики й валідація

AI-агент (Claude Code) пише код швидко і впевнено звітує «усі тести проходять». Сьогодні вчимося **перевіряти** цей результат: як проєкт обмежує агента (дозволи, hook), що сталося в експерименті «одне завдання — два запити» (RSS «Української правди» для `news_hub`) і які «правдоподібні неправди» знайшла рецензія AI-проєкту `depression_dashboard`.

Ключ AI тут **не потрібен**: усі приклади — на справжньому результаті агента, який лежить у папці уроку.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | проєкти і `CLAUDE.md` | 0.1 | |
| 1 | як проєкт обмежує агента: дозволи, hook, skill, довіра до папки | 1.1–1.6 | **Задача 1** |
| 2 | запуск A: код, який агент перевірив лише своїми тестами | 2.1–2.4 | |
| 3 | запуск B і рецензія: що специфікація не питала | 3.1–3.6 | **Задача 2** |
| 4 | `depression_dashboard`: правдоподібна неправда | 4.1–4.8 | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, `class`, константи) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Інтернет і ключі API не потрібні. Числа моделі на синтетичних даних у тебе можуть трохи відрізнятися — важливі висновки.
- Розбір експерименту — на [сторінці уроку](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_43/); довідник — [Claude Code](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/ai/claude_code/).

---
## 0. Проєкти

Клітинка завантажує урок (у Colab), ставить залежності й переходить у папку `news_hub`. Пакет `depression_dashboard` підключаємо за шляхом.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_43_ai_dev_tools"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import fakeredis, fastapi, flask, pytest, sklearn, tzdata  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt",
                    "flask>=3.0.3", "pandas>=2.2.3", "scikit-learn>=1.5.2"], check=True)

LESSON_DIR = os.getcwd()
DASHBOARD = os.path.join(LESSON_DIR, "depression_dashboard")
os.chdir("news_hub")
sys.path.insert(0, os.getcwd())
sys.path.append(DASHBOARD)
os.environ["REDIS_URL"] = "fakeredis://"
print("папка проєкту:", os.getcwd())

### Приклад 0.1. `CLAUDE.md` — пам'ять проєкту для агента

Агент бачить лише те, що є в репозиторії та в запиті. Правила, яких не видно з коду, записано в `CLAUDE.md` — Claude Code читає його на старті кожної сесії. Подивимось на розділ «Правила»:

In [ ]:
claude_md = open("CLAUDE.md", encoding="utf-8").read()
rules = claude_md.split("## Правила")[1].split("\n## ")[0]
print("## Правила" + rules)
print("рядків у файлі:", len(claude_md.splitlines()))

Кожне правило — урок попередніх занять: `endswith` і мережа в тестах — урок 42, «не змінюй тести, щоб вони пройшли» — урок 26. Файл короткий навмисно: його читають щоразу, і він займає місце в контексті агента.

---
## 1. Як проєкт обмежує агента

`CLAUDE.md` — **контекст**: агент зазвичай його виконує, але не гарантовано. Те, що **мусить** виконатися, живе в `.claude/settings.json`: дозволи (`permissions`) і hook-и.

```text
CLAUDE.md, skill   — контекст:    агент зазвичай виконує, але може не виконати
permissions        — межа:        deny / ask / allow перед кожним інструментом
hook               — перевірка:   після кожної правки pytest -m unit; впало → агент виправляє
тести + рецензія   — людина:      що саме має працювати і чого тести не бачать
```

### Приклад 1.1. Дозволи з `.claude/settings.json`

In [ ]:
import json

settings = json.load(open(".claude/settings.json", encoding="utf-8"))
for kind in ("allow", "ask", "deny"):
    print(f"{kind:5}:", ", ".join(settings["permissions"][kind]))
print("hook:", settings["hooks"]["PostToolUse"][0]["matcher"], "→",
      settings["hooks"]["PostToolUse"][0]["hooks"][0]["command"])

| Рядок | Навіщо |
|---|---|
| `allow: pytest, mypy, git diff/status` | агент сам перевіряє себе без запиту до людини |
| `ask: Edit(./tests/**)` | зміна тестів — лише з дозволу: «тести проходять, бо їх змінили» — найчастіший обман |
| `deny: Read(./.env)` | секрети не потрапляють у контекст, а отже й у код чи журнал |
| `deny: pip install` | жодних нових (або вигаданих) пакетів без людини |
| `deny: WebFetch` | чужі сторінки — джерело промпт-ін'єкцій |
| hook `PostToolUse` | після **кожної** правки (`Edit` або `Write`) — скрипт з тестами |

### Приклад 1.2. Hook — це звичайна програма

`.claude/hooks/unit_tests.py` Claude Code запускає після кожної правки. На stdin hook отримує JSON події, а код виходу каже, що робити далі. Подивимось на код:

In [ ]:
print(open(".claude/hooks/unit_tests.py", encoding="utf-8").read())

Не `.py` — одразу вихід з кодом `0`. Інакше — `pytest -m unit -x`; якщо впав — хвіст виводу в stderr і код `2`.

### Приклад 1.3. Запускаємо hook так, як це робить Claude Code

`run_hook(file_path, project)` збирає таку саму подію `PostToolUse` і запускає скрипт у папці проєкту:

In [ ]:
import time


def run_hook(file_path: str, project: str = ".") -> tuple[int, str, float]:
    """Запустити hook для події «агент змінив file_path»; (код виходу, stderr, секунд)."""
    event = {"hook_event_name": "PostToolUse", "tool_name": "Edit", "tool_input": {"file_path": file_path}}
    start = time.perf_counter()
    result = subprocess.run([sys.executable, ".claude/hooks/unit_tests.py"], input=json.dumps(event),
                            capture_output=True, text=True, cwd=project)
    return result.returncode, result.stderr, time.perf_counter() - start

In [ ]:
for path in ("news_hub/rss.py", "README.md"):
    returncode, stderr, seconds = run_hook(path)
    print(f"{path:16} → код {returncode} за {seconds:.1f} с; stderr: {stderr.strip() or '—'}")

Обидва коди `0`: тести зелені, а для `README.md` hook тести й не запускав — тому він майже миттєвий.

**🔮 Прогноз:** агент зламав `news_hub/rss.py` так, що `pytest -m unit` падає. Який код виходу поверне hook і що побачить агент?

<details>
<summary>Відповідь</summary>

`2` і хвіст виводу pytest у stderr. Для `PostToolUse` код 2 не скасовує правку (вона вже відбулась), але Claude Code передає stderr агентові як помилку, яку треба виправити, перш ніж іти далі.

</details>

### Приклад 1.4. Hook ловить зламану правку

Перевіримо прогноз, не чіпаючи проєкту: скопіюємо `news_hub` у тимчасову папку, «зіпсуємо» там `rss.py` так, як міг би агент (час у GMT перестає перераховуватись у київський), і запустимо hook у копії.

In [ ]:
import re
import shutil
import tempfile
from pathlib import Path


def broken_copy() -> Path:
    """Копія news_hub, у якій rss.py «забув» перерахувати час у київський."""
    copy = Path(tempfile.mkdtemp()) / "news_hub"
    shutil.copytree(".", copy, ignore=shutil.ignore_patterns("__pycache__", "*.db", ".coverage"))
    rss = copy / "news_hub" / "rss.py"
    rss.write_text(rss.read_text(encoding="utf-8").replace(".astimezone(KYIV)", ""), encoding="utf-8")
    return copy

In [ ]:
copy = broken_copy()
returncode, stderr, seconds = run_hook("news_hub/rss.py", project=str(copy))
print("код виходу:", returncode)
plain = re.sub(r"\x1b\[[0-9;]*m", "", stderr)            # прибрати кольори терміналу
print("\n".join(line for line in plain.splitlines() if line.startswith(("pytest -m unit", "FAILED", "E  ")) or " failed" in line))
shutil.rmtree(copy.parent)
assert returncode == 2

Агент отримує не «щось пішло не так», а назву тесту, що впав, і очікуване значення. Головне — це відбувається **автоматично** після кожної правки, навіть якщо агент «забув» запустити тести.

### Приклад 1.5. Skill — інструкція на виклик

Skill `/add-news-source` записує порядок роботи з новим джерелом. Його викликають командою, і тоді текст потрапляє в контекст агента:

In [ ]:
skill = open(".claude/skills/add-news-source/SKILL.md", encoding="utf-8").read()
front, body = skill.split("---")[1], skill.split("---", 2)[2]
print(front.strip())
print("кроків:", sum(line[:2].strip().rstrip(".").isdigit() for line in body.splitlines()))

`disable-model-invocation: true` — агент не викличе skill сам, лише людина командою. `!`git status --short`` у кінці файлу — команда, вивід якої Claude Code підставить у текст при виклику.

### Задача 1. Хто вирішує: `allow`, `ask` чи `deny`

Напиши спрощений `decide(command)` для правил виду `Bash(префікс *)` з `.claude/settings.json`. Розбиття команди і перевірку «частина під правилом» уже написано в наступній клітинці (`parts`, `covered`):

- **кожна** частина складеної команди (`&&`, `||`, `;`, `|`) перевіряється окремо;
- хоч одна частина під `deny` → `"deny"`;
- **усі** частини під `allow` → `"allow"`;
- інакше (хоч одна під `ask` або не під жодним правилом) → `"ask"`.

In [ ]:
import re


def bash_prefixes(rules: list[str]) -> list[str]:
    """['Bash(pytest *)', 'WebFetch'] → ['pytest']: лише правила Bash, без « *»."""
    return [m.group(1) for rule in rules if (m := re.fullmatch(r"Bash\((.+?)(?: \*)?\)", rule))]


ALLOW, ASK, DENY = (bash_prefixes(settings["permissions"].get(key, [])) for key in ("allow", "ask", "deny"))


def covered(part: str, prefixes: list[str]) -> bool:
    """Частина дорівнює префіксу або починається з «префікс + пробіл»."""
    return any(part == p or part.startswith(p + " ") for p in prefixes)


def split_command(command: str) -> list[str]:
    return [p.strip() for p in re.split(r"&&|\|\||;|\|", command) if p.strip()]

In [ ]:
print("allow:", ALLOW, "| deny:", DENY)
print(split_command("git status && git push origin main | cat"))
print(covered("pytest -m unit", ALLOW), covered("pytestx", ALLOW))

In [ ]:
def decide(command: str) -> str:
    parts = split_command(command)
    # BEGIN SOLUTION
    if any(covered(p, DENY) for p in parts):
        return "deny"
    if all(covered(p, ALLOW) for p in parts):
        return "allow"
    return "ask"
    # END SOLUTION

In [ ]:
cases = {"pytest -m unit": "allow", "mypy --strict news_hub": "allow", "git push origin main": "deny",
         "pip install requests": "deny", "pytest && rm -rf .": "ask", "git status && git push": "deny"}
assert decide("pytest -m unit") == "allow", "pytest є в allow — має бути 'allow'"
for command, expected in cases.items():
    print(f"{command:28} → {decide(command)}")
    assert decide(command) == expected, command
print("✅ Задача 1 розв'язана")

Справжній Claude Code перевіряє складніше (обгортки `timeout`, підоболонки, змінні), але принцип той самий: `deny` сильніший за `allow`, і складена команда не «прослизне» за префіксом першої частини.

### Приклад 1.6. `allow` з репозиторію не діє без довіри

Знахідка уроку: перший запуск агента в налаштованій папці жодного разу не зміг запустити `pytest` — «This command requires approval», хоча `Bash(pytest *)` стоїть в `allow`. Claude Code пояснив у stderr:

```text
Ignoring 5 permissions.allow entries from .claude/settings.json: this workspace has not been trusted.
```

`allow` з файлу репозиторію діє лише після того, як ти **довірив папку** (діалог при першому інтерактивному запуску). `deny` і `ask` діють одразу. Інакше будь-який клонований репозиторій сам собі дозволив би команди. Змоделюємо це правило:

In [ ]:
def effective_rules(permissions: dict, trusted: bool) -> dict:
    """Які правила з файлу репозиторію справді діють."""
    return {kind: (rules if kind != "allow" or trusted else [])
            for kind, rules in permissions.items()}

In [ ]:
for trusted in (False, True):
    rules = effective_rules(settings["permissions"], trusted)
    print(f"довірена папка: {trusted!s:5} → allow {len(rules['allow'])}, ask {len(rules['ask'])}, deny {len(rules['deny'])}")

Для чесного порівняння обидва запуски експерименту нижче отримали однакові дозволи прапорцем `--allowedTools`.

---
## 2. Запуск A: зелений на власних тестах

Завдання обом запускам: додати друге джерело — RSS «Української правди». Запит A був коротким: «Додай нове джерело… новини мають зберігатися так само, як з rbc.ua». Стрічка з середовища недоступна, тож агент A **вигадав** власний зразок RSS — лише з датами `+0300` — і написав тести на ньому. Звіт: «Test suite passes (101/101)».

### Приклад 2.1. Функція часу запуску A — дослівно

In [ ]:
from email.utils import parsedate_to_datetime


def pravda_time_run_a(pub_date: str) -> str:
    """RFC 822 (`<pubDate>`) → «HH:MM:SS», як час у стрічці rbc.ua; не розібрали — порожній рядок."""
    if not pub_date:
        return ""
    try:
        return parsedate_to_datetime(pub_date).strftime("%H:%M:%S")
    except (TypeError, ValueError):
        return ""

In [ ]:
print(pravda_time_run_a("Sat, 26 Sep 2026 21:40:00 +0300"))

На даті з `+0300` — правильно. Але `strftime` бере час **як записано**, не перераховуючи пояс.

### Приклад 2.2. Ті самі дати — у фінальному `rss.py`

Фінальний `news_hub.rss._kyiv_datetime` перераховує будь-який пояс у київський. Порівняємо обидві функції на кількох форматах `pubDate`:

In [ ]:
from news_hub.rss import _kyiv_datetime

PUB_DATES = ["Sat, 26 Sep 2026 21:40:00 +0300", "Sat, 26 Sep 2026 17:05:00 GMT",
             "Sat, 26 Sep 2026 17:05:00 -0000", "Sat, 26 Jan 2026 10:00:00 +0000", "garbage", ""]

In [ ]:
for pub_date in PUB_DATES:
    final = _kyiv_datetime(pub_date)
    print(f"{pub_date!r:36} запуск A: {pravda_time_run_a(pub_date)!r:11} фінальний: {final!r}")

- `17:05 GMT` у Києві — `20:05` (літній час, +3), а `10:00 +0000` у січні — `12:00` (зимовий, +2). Запуск A записував лондонський час як київський;
- `-0000` за RFC 2822 — «пояс невідомий», `parsedate_to_datetime` дає datetime **без** поясу; фінальна версія вважає його UTC, а не часом сервера;
- битий і порожній рядок — новина без часу, а не помилка на всю стрічку.

Тести агента цього не бачили: у його фікстурі всі дати були з `+0300`.

### Приклад 2.3. «Тиха» відмова, закріплена тестом

Тест із запуску A: `assert parse_pravda_rss("не xml") == []`. Ось поведінка коду A в мініатюрі і перевірка-**вимога**: на битому XML парсер має кинути `ValueError`.

In [ ]:
import xml.etree.ElementTree as ET

from news_hub.rss import parse_pravda_rss


def parse_run_a(xml: str) -> list:
    try:
        root = ET.fromstring(xml)
    except ET.ParseError:
        return []
    return [item for item in root.iter("item")]


def raises_on_garbage(parse) -> bool:
    """Вимога: обрізаний XML — ValueError, а не «порожня стрічка»."""
    try:
        parse("<rss><channel><item><title>обрізано")
    except ValueError:
        return True
    return False

In [ ]:
print("тест запуску A на коді A:", parse_run_a("не xml") == [])
print("вимога — запуск A:", raises_on_garbage(parse_run_a), "| фінальний:", raises_on_garbage(parse_pravda_rss))
try:
    parse_pravda_rss("<rss><channel><item><title>обрізано")
except ValueError as error:
    print("фінальний:", error)

Тест A зелений і покриває гілку `except` — покриття 100%. Але він **закріплює** помилку: зламана стрічка, обрізана відповідь, HTML замість XML — усе стає «порожньою стрічкою», і сигналу, що джерело зламалось, не буде ніколи. Агент написав тест з тим самим припущенням, що й код, — тест перевіряє реалізацію, а не вимогу.

### Приклад 2.4. Що ще зробив запуск A

| Знахідка | Чому тести агента її не бачили |
|---|---|
| `17:05` для `pubDate` у GMT — час Лондона | у його фікстурі всі дати з `+0300` |
| новину з `epravda.com.ua` відхилено | стрічки не бачив — не знав про сестринські сайти |
| битий XML → `[]` | тест закріплює це як правильну поведінку |
| вигадані «новини» реального видання у фікстурі без позначки | — |
| `pravda.com.ua` дописано в сторінки для `POST /api/scrape` | «зберігати так само» прочитано як «збирати так само» |
| змінено 6 файлів наявних тестів | — |

Перевіримо останнє на фінальному коді: `POST /api/scrape` і далі приймає лише rbc.ua, а **модель** новини — три домени.

In [ ]:
from pydantic import ValidationError

from news_hub.api import ScrapeRequest
from news_hub.models import is_allowed_host, is_rbc_host

for host in ("www.rbc.ua", "www.pravda.com.ua", "www.epravda.com.ua", "fakepravda.com.ua", "pravda.com.ua.evil.com"):
    try:
        ScrapeRequest(pages=[f"https://{host}/"])
        scrape = "прийнято"
    except ValidationError:
        scrape = "відхилено"
    print(f"{host:24} новина: {is_allowed_host(host)!s:5}  POST /api/scrape: {scrape}")

Дві різні функції для двох різних рішень: **звідки може бути новина** (`is_allowed_host`) і **куди сервер може піти за сторінкою** (`is_rbc_host`). Розширити друге «заодно» — відкрити сервер для завантаження чужих адрес.

---
## 3. Запуск B і рецензія

Запуск B отримав **специфікацію**: `tests/unit/test_pravda.py` (18 тестів, написаних до коду) і зразок стрічки `tests/fixtures/pravda_rss.xml`, а в запиті — «зроби так, щоб проходили всі тести і `mypy --strict`, не змінюючи тестів; наприкінці перелічи рішення, які ти ухвалив сам». Результат — удвічі швидше, 2 файли замість 10, тести не змінено.

### Приклад 3.1. Специфікація і тести рецензента

In [ ]:
def run_pytest(*args: str, cwd: str = ".") -> str:
    """pytest окремим процесом; повертає вивід."""
    result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", *args],
                            capture_output=True, text=True, cwd=cwd)
    return result.stdout + result.stderr

In [ ]:
for path in ("tests/unit/test_pravda.py", "tests/unit/test_pravda_review.py"):
    print(f"{path:35}", run_pytest(path).strip().splitlines()[-1])
summary = run_pytest("tests/unit/test_pravda.py", "tests/unit/test_pravda_review.py").strip().splitlines()[-1]
print("разом:", summary)
assert summary.startswith("21 passed")

18 тестів специфікації + 3 тести рецензента. На коді агента B три тести рецензента **падали** — він виконав специфікацію, але вона мовчала саме про ці випадки (приклад 3.4).

### Приклад 3.2. Що всередині специфікації

Назви тестів — це вже перелік вимог (8 функцій; з `parametrize` — 18 тестів):

In [ ]:
import ast


def test_names(path: str) -> list[str]:
    tree = ast.parse(open(path, encoding="utf-8").read())
    return [node.name for node in tree.body if isinstance(node, ast.FunctionDef) and node.name.startswith("test_")]

In [ ]:
for name in test_names("tests/unit/test_pravda.py"):
    print("-", name)

Порядок і дублікати, чисті поля, київський час, конвеєр «RSS → модель», межові домени, `POST /api/scrape` лише для rbc.ua, заборона мережі, битий XML → `ValueError`. Кожне рішення агента або перевірене тестом, або назване у звіті.

### Приклад 3.3. Стрічка → модель: що прийнято і що відхилено

In [ ]:
from news_hub.models import validate_news

feed = open("tests/fixtures/pravda_rss.xml", encoding="utf-8").read()
raw_items = parse_pravda_rss(feed)
valid, rejected = validate_news(raw_items)
print("у стрічці <item>:", feed.count("<item>"), "| після парсера:", len(raw_items))
for item in valid:
    print(item.source, item.lang, item.category, item.published_time, "|", item.title)
print("відхилено:", [(r.raw["title"], r.errors) for r in rejected])

П'ять `<item>` → чотири після парсера (дубль за адресою прибрано) → три прийнято. `&quot;` став лапками, `epravda.com.ua` — прийнято з правильним київським часом `20:05`, а «Відео дня» відхилено з причиною — нічого не зникло мовчки.

### Приклад 3.4. Рецензія: «що буде, якщо…»

Зелена специфікація — не кінець роботи. Рецензент питає про те, чого специфікація не питала. Зберемо стрічку з однієї новини і змінюватимемо по одному тегу:

In [ ]:
ITEM = ('<item><title>Уряд оновив правила вступу до університетів</title>'
        '<link>https://www.pravda.com.ua/news/2026/09/26/{n}/</link>{extra}</item>')


def one_item_feed(extra: str, n: int = 7501001) -> str:
    return "<rss><channel>" + ITEM.format(n=n, extra=extra) + "</channel></rss>"


def review(extra: str) -> str:
    """Що станеться з новиною, у якої додаткові теги extra."""
    valid, rejected = validate_news(parse_pravda_rss(one_item_feed(extra)))
    if rejected:
        return f"відхилено: {rejected[0].errors}"
    return f"категорія {valid[0].category!r}, час {valid[0].published_time}"

In [ ]:
CASES = {
    "усе є": "<category>Суспільство</category><pubDate>Sat, 26 Sep 2026 21:40:00 +0300</pubDate>",
    "немає <category>": "<pubDate>Sat, 26 Sep 2026 21:40:00 +0300</pubDate>",
    "немає <pubDate>": "<category>Новини</category>",
    "битий <pubDate>": "<category>Новини</category><pubDate>garbage</pubDate>",
}
for name, extra in CASES.items():
    print(f"{name:17} → {review(extra)}")

| Що буде, якщо… | Код агента B | Після рецензії |
|---|---|---|
| немає `<category>` | категорія з URL: `/news/2026/09/26/…` → **«2026»** | «Новини» |
| немає чи битий `<pubDate>` | `ValueError` — і **вся стрічка** губиться через одну новину | ця новина без часу, решта — як є |
| Windows без пакета `tzdata` | `ZoneInfoNotFoundError` під час `import news_hub.rss` | `tzdata` у `requirements.txt` |

Кожен рядок — тест у `tests/unit/test_pravda_review.py`.

### Приклад 3.5. Третій випадок: інша ОС

У Windows немає системної бази часових поясів — `ZoneInfo("Europe/Kyiv")` бере її з пакета `tzdata`. Тест рецензента імітує це: `PYTHONTZPATH` вказує в порожню папку, і модуль імпортується в окремому процесі.

In [ ]:
def import_rss_without_system_tz() -> str:
    env = {**os.environ, "PYTHONTZPATH": tempfile.mkdtemp()}
    result = subprocess.run([sys.executable, "-c", "import news_hub.rss as r; print(r.KYIV)"],
                            capture_output=True, text=True, env=env)
    return (result.stdout or result.stderr).strip().splitlines()[-1]

In [ ]:
import importlib.util

print("tzdata встановлено:", importlib.util.find_spec("tzdata") is not None)
print("імпорт без системної бази:", import_rss_without_system_tz())

З пакетом `tzdata` модуль імпортується і без системної бази. Без нього на Windows `news_hub.rss` не імпортувався б **узагалі** — а на Linux розробника все працює. «Інша ОС» — окремий пункт чекліста рецензії.

**🔮 Прогноз:** сайт віддав замість RSS сторінку помилки `<html><body><h1>403</h1></body></html>` (правильний XML, код 200). Що поверне `parse_pravda_rss`?

<details>
<summary>Відповідь</summary>

`[]` — так само, як на порожній стрічці: в HTML немає жодного `<item>`. Збір «успішний», новин 0, помилки ніхто не побачить. Правильний XML — не обов'язково RSS.

</details>

### Приклад 3.6. Сторінка помилки замість стрічки

In [ ]:
print("HTML-сторінка помилки →", parse_pravda_rss("<html><body><h1>403 Forbidden</h1></body></html>"))
print("порожня стрічка       →", parse_pravda_rss("<rss><channel></channel></rss>"))

Два різні стани — один результат. Порожня стрічка — легальний стан («нових новин немає»), а сторінка помилки — збій джерела, який має бути видно.

### Задача 2. Строгий парсер: RSS або `ValueError`

Напиши `parse_strict(xml)` — обгортку над `parse_pravda_rss`:

- розбери документ `ET.fromstring(xml)`;
- якщо корінь — не `<rss>` (`root.tag != "rss"`) **або** в ньому немає `<channel>` (`root.find("channel") is None`) — `raise ValueError(...)` з поясненням, який тег отримали;
- інакше поверни `parse_pravda_rss(xml)`. Порожня стрічка `<rss><channel></channel></rss>` — легальна: `[]`.

In [ ]:
# BEGIN SOLUTION
def parse_strict(xml: str) -> list:
    root = ET.fromstring(xml)
    if root.tag != "rss" or root.find("channel") is None:
        raise ValueError(f"очікуємо RSS (<rss><channel>), а отримали <{root.tag}>")
    return parse_pravda_rss(xml)
# END SOLUTION

In [ ]:
assert parse_strict("<rss><channel></channel></rss>") == [], "порожня стрічка — легальна: []"
for bad in ("<html><body><h1>403 Forbidden</h1></body></html>", "<rss><title>без channel</title></rss>"):
    try:
        parse_strict(bad)
        raise AssertionError(f"{bad[:20]}… мав дати ValueError")
    except ValueError as error:
        print("ValueError:", error)
assert len(parse_strict(feed)) == 4
print("✅ Задача 2 розв'язана")

---
## 4. `depression_dashboard`: правдоподібна неправда

AI отримав рольові промпти («You are a Senior Python Architect and Data Scientist…», 12 етапів від EDA до Docker) і згенерував Flask API + Streamlit-дашборд + RandomForest, KMeans, IsolationForest. Застосунок запускався, усі ендпоінти відповідали `200`, дашборд малював графіки. Тестів у промптах не просили.

> ⚠️ **Чутлива тема.** Застосунок оцінює ризик депресії. У курсі це **кейс рецензії коду**, а не інструмент: модель на відкритому датасеті не є діагнозом.

### Приклад 4.1. Бекенд на синтетичних даних

Справжній датасет у репозиторій не входить. Генератор синтетичних даних з колонками справжнього — у `tests/conftest.py` проєкту; підключаємо його за шляхом і направляємо бекенд у тимчасову папку (дані й файл моделі).

In [ ]:
import logging
import tempfile

spec = importlib.util.spec_from_file_location("dd_conftest", os.path.join(DASHBOARD, "tests", "conftest.py"))
dd_conftest = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dd_conftest)

data_dir = Path(tempfile.mkdtemp())
dd_conftest.make_students(600).to_csv(data_dir / "Student Depression Dataset.csv", index=False)
dd_conftest.make_burnout().to_csv(data_dir / "mental_health_burnout_tech_2026.csv", index=False)

from backend.config import config
from backend.services import data_service, ml_service

config.STUDENT_DATASET = data_dir / "Student Depression Dataset.csv"
config.BURNOUT_DATASET = data_dir / "mental_health_burnout_tech_2026.csv"
ml_service.MODEL_PATH = data_dir / "models" / "classifier.pkl"
data_service.load_student_df.cache_clear()
logging.disable(logging.INFO)                  # журнал бекенду (INFO) не друкуємо

from backend.app import create_app

dd = create_app().test_client()

In [ ]:
print(dd.get("/health").get_json(), "| рядків:", len(data_service.load_student_df()))
print("колонки:", list(data_service.load_student_df().columns)[:8], "…")

`create_app().test_client()` — тестовий клієнт Flask: запити без сервера й мережі, як `Client` у Django чи `TestClient` у FastAPI.

### Приклад 4.2. Вада, яку не видно на скриншоті: порядок ключів JSON

Сервіс повертав важливість ознак, відсортовану за спаданням, а дашборд брав перші 12. Але `jsonify` у Flask за замовчуванням **сортує ключі за абеткою**:

In [ ]:
from flask import Flask, jsonify

demo_importance = {"Suicidal_enc": 0.27, "Risk_Score": 0.14, "Academic Pressure": 0.11, "Age": 0.03}
demo_app = Flask("demo")
with demo_app.app_context():
    print("Flask за замовчуванням:", jsonify(demo_importance).get_data(as_text=True).strip())
    demo_app.json.sort_keys = False
    print("sort_keys = False:     ", jsonify(demo_importance).get_data(as_text=True).strip())

Тепер — справжня відповідь бекенду (з `StrictJSONProvider(sort_keys=False)`) і те, що отримав би клієнт від стандартного `jsonify`. Перший запит навчає модель — це займе кілька секунд.

In [ ]:
importance = dd.get("/api/feature-importance").get_json()
old_app = Flask("old")
with old_app.app_context():
    old_order = list(json.loads(jsonify(importance).get_data(as_text=True)))
print("виправлений бекенд:", list(importance)[:4])
print("старий jsonify:    ", old_order[:4])
values = list(importance.values())
assert values == sorted(values, reverse=True) and old_order == sorted(importance)

Графік «Top Feature Importances» показував перші 12 ознак **за абеткою**. Жодної помилки, жодного попередження, гарний графік. Правильно — `sort_keys = False` у власному JSON-провайдері, а UI сортує сам, не покладаючись на порядок ключів.

### Приклад 4.3. `/api/predict` приймає лише анкету

Раніше сервер брав будь-який JSON: зайві поля мовчки ігнорувались, похідні ознаки (`Risk_Score`) приходили від клієнта, `age=-500` давав `200`. Тепер вхід перевіряє Pydantic-модель `StudentProfile` з `extra="forbid"`. Надішлемо правильну анкету й кілька поганих:

In [ ]:
PROFILE = {"age": 22, "gender": "Female", "sleep_duration": "5-6 hours", "academic_pressure": 3,
           "work_pressure": 0, "study_satisfaction": 3, "job_satisfaction": 0, "work_study_hours": 6,
           "financial_stress": 2, "dietary_habits": "Moderate", "family_history": False, "suicidal_thoughts": False}

BAD_INPUTS = {
    "age = -500": {"age": -500},
    "financial_stress = 999": {"financial_stress": 999},
    "sleep = Others": {"sleep_duration": "Others"},
    "підроблений Risk_Score": {"Risk_Score": -100},
    "повзунок CGPA": {"CGPA": 7.5},
}

In [ ]:
print("анкета →", dd.post("/api/predict", json=PROFILE).get_json())
for name, change in BAD_INPUTS.items():
    response = dd.post("/api/predict", json={**PROFILE, **change})
    print(f"{name:24} → {response.status_code} {[e['field'] for e in response.get_json()['errors']]}")

Кожна погана анкета — `422` з назвою поля. Поле `CGPA` (повзунок старого UI) модель **не знала**: повзунок ні на що не впливав, а тепер це помилка, а не тиша. Похідну ознаку `Risk_Score` рахує сервер — підробити прогноз полем вже не вийде.

### Приклад 4.4. Пропущене поле: `422` без нутрощів

In [ ]:
profile = dict(PROFILE)
profile.pop("age")
response = dd.post("/api/predict", json=profile)
text = response.get_data(as_text=True)
print(response.status_code, response.get_json()["errors"])
print("у відповіді є Traceback / KeyError?", "Traceback" in text or "KeyError" in text)
print("тіло-список →", dd.post("/api/predict", json=[1, 2]).status_code)

Раніше пропущене поле давало `500` з текстом винятку pandas у відповіді — і клієнт бачив внутрішню будову сервера. Тепер — `422` з переліком полів.

### Приклад 4.5. Назви кластерів: вшиті проти обчислених

Старий UI називав кластери словником у коді: `{0: "High Pressure", 1: "Sleep Deprived", …}`. Але номери кластерів KMeans **довільні**. Знайдемо кластер, що найменше спить, і порівняємо стару назву з новою (яку дає профіль кластера):

In [ ]:
OLD_LABELS = {0: "High Pressure", 1: "Sleep Deprived", 2: "Stable / Low Risk", 3: "Financially Stressed"}
clusters = dd.get("/api/clusters").get_json()

In [ ]:
profiles = clusters["cluster_profiles"]
for cid in sorted(profiles):
    print(f"кластер {cid}: сон {profiles[cid]['Sleep_hours']:.2f} год, розмір {clusters['cluster_sizes'][cid]:3} | "
          f"стара назва: {OLD_LABELS[int(cid)]:20} | нова: {clusters['cluster_names'][cid]}")
least_sleep = min(profiles, key=lambda cid: profiles[cid]["Sleep_hours"])
print("найменше сну — кластер", least_sleep, "| стара назва:", OLD_LABELS[int(least_sleep)])

Подивись на таблицю: назву «Sleep Deprived» старий UI дав кластеру, який спить не менше за інших, а кластер з найменшим сном називається інакше. Вшита назва вгадує лише випадково (у книзі на моделі зі справжнього датасету «Sleep Deprived» спав **найбільше** — 8,15 год). Нова назва — «ознака, де кластер найдалі від середнього» — завжди відповідає профілю.

### Приклад 4.6. `POST /api/train` — лише з токеном адміністратора

Згенерований код дозволяв будь-кому перенавчити й перезаписати модель. Тепер потрібен заголовок `X-Admin-Token`, що збігається з `ADMIN_TOKEN` зі змінної середовища (`hmac.compare_digest` — порівняння без витоку часу); без змінної ендпоінт вимкнено.

In [ ]:
from unittest import mock

print("ADMIN_TOKEN не задано, без заголовка →", dd.post("/api/train").status_code)
with mock.patch.object(config, "ADMIN_TOKEN", "test-token-" + "x" * 20):
    print("неправильний токен →", dd.post("/api/train", headers={"X-Admin-Token": "wrong"}).status_code)
    ok = dd.post("/api/train", headers={"X-Admin-Token": "test-token-" + "x" * 20})
    print("правильний токен   →", ok.status_code, "| AUC:", ok.get_json()["metrics"]["auc"])

`mock.patch.object(config, "ADMIN_TOKEN", …)` — тимчасово задаємо токен лише всередині `with`, як `monkeypatch` у тесті.

### Приклад 4.7. Модель з диска — один раз

Файл моделі (pickle на 57 МБ у справжньому датасеті) читався з диска на **кожен** запит. Порахуємо виклики `pickle.load` за три прогнози — шпигуном `wraps=` (урок 42):

In [ ]:
import pickle

dd.post("/api/predict", json=PROFILE)                         # модель уже в пам'яті
with mock.patch("pickle.load", wraps=pickle.load) as spy:
    for _ in range(3):
        dd.post("/api/predict", json=PROFILE)
print("читань з диска за 3 прогнози:", spy.call_count)

Нуль: модель у пам'яті, кеш скидається лише після навчання. Окремо: `pickle` виконує код під час читання — завантажувати можна лише власні, довірені файли.

### Приклад 4.8. Тести рецензії цілком

Кожна знахідка рецензії — тест у `depression_dashboard/tests/test_review.py`. На згенерованому коді всі 17 падали, на копії уроку:

In [ ]:
dashboard_report = run_pytest("-p", "no:logging", cwd=DASHBOARD)
print(next(line for line in reversed(dashboard_report.splitlines()) if " passed" in line or " failed" in line))

| # | Знахідка | Приклад |
|---|---|---|
| 1 | «топ-ознаки» — за абеткою | 4.2 |
| 2 | назви кластерів вшиті в UI | 4.5 |
| 3–4 | `/api/predict` приймав похідні ознаки, `age=-500`, мертвий повзунок CGPA, `500` з pandas | 4.3–4.4 |
| 5 | `POST /api/train` без захисту | 4.6 |
| 6 | витік даних: імпутер навчався на всіх рядках до `train_test_split` | книга |
| 7 | модель з диска на кожен запит | 4.7 |
| 8 | кешований DataFrame змінювали на місці | задача 3 |
| 9 | `NaN` у відповіді — невалідний JSON | книга |
| 10 | папка `ui/pages/` додавала 5 порожніх сторінок у меню Streamlit | книга |

Жодна з вад не падає з помилкою. Кожна дає **правдоподібну неправду** — це головний ризик AI-коду.

---
## Знайди помилку

У старому `get_depression_by_group()` кешований DataFrame змінювали на місці: `/api/groups` дописував колонку, і після нього в `/api/summary` з'являлась чужа колонка. Ось той самий патерн у мініатюрі: `load()` кешує DataFrame (`lru_cache`), а `groups()` дописує в нього колонку `bucket`.

In [ ]:
from functools import lru_cache

import pandas as pd


@lru_cache(maxsize=1)
def load() -> pd.DataFrame:
    return pd.DataFrame({"stress": [1, 3, 5], "depression": [0, 1, 1]})


def groups_old() -> dict:
    df = load()
    df["bucket"] = pd.cut(df["stress"], bins=[0, 2, 5], labels=["low", "high"])
    return df.groupby("bucket", observed=True)["depression"].mean().to_dict()

In [ ]:
print(groups_old())
print("колонки кешу після groups_old():", list(load().columns))
load.cache_clear()                              # повертаємо чистий кеш для задачі

`lru_cache` повертає **той самий** об'єкт щоразу — зміна «локальної» `df` змінила кеш для всіх, хто його читає.

### Задача 3. Кеш лишається незмінним

Напиши `groups()` — те саме, що `groups_old()`, але без зміни кешованого DataFrame: працюй з **копією** (`load().copy()`). Результат має бути той самий: `{'low': 0.0, 'high': 1.0}`.

In [ ]:
def groups() -> dict:
    # BEGIN SOLUTION
    df = load().copy()
    df["bucket"] = pd.cut(df["stress"], bins=[0, 2, 5], labels=["low", "high"])
    return df.groupby("bucket", observed=True)["depression"].mean().to_dict()
    # END SOLUTION

In [ ]:
result = groups()
assert isinstance(result, dict), "groups() має повернути словник {група: частка}"
assert list(load().columns) == ["stress", "depression"], "groups() змінив кешований DataFrame — працюй з копією"
print(result, "| колонки кешу:", list(load().columns))
assert result == {"low": 0.0, "high": 1.0}
print("✅ Задача 3 розв'язана")

---
## Самоперевірка

1. Чим `CLAUDE.md` відрізняється від `permissions.deny`?
2. Чому 101 зелений тест запуску A не доводить правильності коду?
3. Що дала специфікація запуску B і чого не дала?
4. Чому вада з `jsonify` не видна на скриншоті дашборду?
5. Три пункти з чекліста рецензії AI-коду, які ти застосуєш у своєму проєкті першими.

<details>
<summary>Відповіді</summary>

1. `CLAUDE.md` — контекст: агент зазвичай виконує правила, але може не виконати. `deny` — межа: заборонену дію Claude Code не виконає, хоч би що вирішив агент (приклади 1.1, задача 1).
2. Тести написав той самий агент з тими самими припущеннями: вигадана фікстура лише з `+0300` і тест, що закріплює «тиху» відмову (приклади 2.2–2.3).
3. Дала перевірний результат: кожне рішення перевірене тестом або назване у звіті, тести не змінено. Не дала — відповідей на питання, яких у ній не було: категорія без `<category>`, битий `pubDate`, інша ОС (приклади 3.4–3.5).
4. Графік виглядає правильно — стовпчики відсортовані, просто це не ті 12 ознак. Порядок ключів губиться в серіалізації JSON, без помилки (приклад 4.2).
5. Наприклад: спершу специфікація (тести, що падають); діф тестів окремо; «що буде, якщо…» — порожнє, відсутнє, битий формат, інший часовий пояс, інша ОС; нічого не пропадає мовчки.

</details>

## Далі

- Книга курсу: [Урок 43](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_43/) — експеримент з двома запусками, рецензія, десять вад дашборду, чекліст.
- Спробуй сам: встанови Claude Code ([довідник](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/ai/claude_code/)), відкрий `news_hub` і виконай `/add-news-source` для третього джерела.
- Урок 44 — Gemini в агрегаторі: відповідь LLM — теж дані, які перевіряє Pydantic.